In [ ]:
# import library
from typing import List
import numpy as np
import torch
import evaluate
from sklearn . model_selection import train_test_split
import nltk
nltk.download ('treebank')


# load tree bank dataset
tagged_sentences = nltk.corpus.treebank.tagged_sents()
print (" Number of samples :", len(tagged_sentences))

# save sentences and tags
sentences, sentence_tags =[], []
for tagged_sentence in tagged_sentences:
    sentence, tags = zip(*tagged_sentence)
    sentences.append ([word.lower() for word in sentence])
    sentence_tags.append ([tag for tag in tags])


In [2]:
train_sentences , test_sentences , train_tags , test_tags = train_test_split(sentences, sentence_tags, test_size =0.3)

valid_sentences , test_sentences , valid_tags , test_tags = train_test_split(test_sentences, test_tags, test_size =0.5)


In [4]:

# tokenization
from transformers import AutoTokenizer
from torch.utils.data import Dataset

model_name = "QCRI/bert-base-multilingual-cased-pos-english"

tokenizer = AutoTokenizer . from_pretrained(model_name, use_fast = True)
MAX_LEN = 256

class PosTagging_Dataset(Dataset):
    def __init__(self, sentences: List[List[str]], tags: List[List[str]], tokenizer, label2id, max_len = MAX_LEN):
        super().__init__()
        self . sentences = sentences
        self . tags = tags
        self . max_len = max_len
        self . tokenizer = tokenizer
        self . label2id = label2id

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx ):
        input_token = self.sentences[idx]
        label_token = self.tags[idx]

        input_token = self.tokenizer.convert_tokens_to_ids(input_token)
        attention_mask = [1] * len(input_token)
        labels = [self.label2id[token] for token in label_token]

        return {"input_ids": self.pad_and_truncate(input_token, pad_id = self.tokenizer.pad_token_id),
                "labels": self.pad_and_truncate(labels, pad_id = label2id["0"]),
                "attention_mask": self.pad_and_truncate(attention_mask, pad_id =0)}

    def pad_and_truncate(self, inputs: List[int], pad_id: int):
        if len(inputs) < self.max_len:
            padded_inputs = inputs + [pad_id] * (self.max_len - len(inputs))
        else :
            padded_inputs = inputs[:self.max_len]
        return torch.as_tensor(padded_inputs)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

c:\Users\ADMIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--QCRI--bert-base-multilingual-cased-pos-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ADMIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarnin

config.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [15]:
from transformers import AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(model_name)


pytorch_model.bin:   0%|          | 0.00/712M [00:00<?, ?B/s]

c:\Users\ADMIN\anaconda3\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of the model checkpoint at QCRI/bert-base-multilingual-cased-pos-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing 